# Lab 4 — Correlation and linear regression: measuring and modelling a relationship
**Data Analysis — Master AI / Master RSD** · Chapter 3: Statistical analysis (correlation, linear regression, interpretation)

*Measure the direction and strength of a relationship, fit it with a line, and judge honestly what the model explains.*

**Lab objectives**
1. **Compute and interpret** Pearson's correlation coefficient $r$ (sign, strength).
2. **Fit** a simple linear regression: by hand (least squares), then with `LinearRegression`.
3. **Evaluate** the model with $R^2$, use it to **predict** — without extrapolating.
4. **Recognise the pitfalls** of $r$: non-linear relationship, outlier, correlation ≠ causation.
5. **Read a correlation matrix** on realistic data and spot multicollinearity.

| Duration | Organisation | Data | Deliverable |
|---|---|---|---|
| 2 sessions of 2 h (4 h total) | Pairs · Jupyter notebook | `study_grades.csv` (course example) · `clients_enriched.csv` (Lab 3) | Completed notebook + conclusions |

⭐ **Continuity:** `study_grades.csv` is the "study hours / grade" example from chapter 3. `clients_enriched.csv` is the **Lab 3** file (988 clients, cleaned and enriched).

**How to use**: complete the cells marked `# TODO`, then run the **✅ Self-check** cell that follows. Answer the questions in the **✍️** cells.

---
## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 20)

---
## Step 1 — Measuring a relationship: Pearson's correlation
*Pearson's correlation, written $r$, measures the direction and strength of a **linear relationship** between two quantitative variables* (section 1).

### 1.0 Load the file
Eight students: number of **study hours per week** and **grade obtained** (out of 20).

In [ ]:
grades = ...   # TODO: pd.read_csv("study_grades.csv") — separator "," (the default)

print(grades.shape)
grades

✅ **Self-check**

In [ ]:
assert grades.shape == (8, 2), f"Expected (8, 2), got {grades.shape}"
assert list(grades.columns) == ["hours", "grade"]
print("✅ 1.0 correct")

### 1.1 Always look first: the scatter plot
Plot the grade ($Y$, target variable) against study hours ($X$, explanatory variable).

In [ ]:
X = grades["hours"]   # explanatory variable
y = grades["grade"]   # target variable

# TODO: scatter plot of y against X (plt.scatter), with a title and axis labels

✍️ **Question:** before any computation, what does the plot say? Direction of the relationship? Are the points roughly aligned?

…

### 1.2 Compute $r$
$$r = \frac{\sum_{i=1}^{n} (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum_{i=1}^{n} (x_i - \bar{x})^2}\;\sqrt{\sum_{i=1}^{n} (y_i - \bar{y})^2}}$$
1. Compute `r_manual` **by hand** with the formula.
2. Compute `r_pandas` with `X.corr(y)` and check that you get the same thing.

In [ ]:
ex = X - X.mean()   # deviations from the mean of X
ey = y - y.mean()   # deviations from the mean of y

r_manual = ...   # TODO: Pearson's formula with ex and ey (.sum(), np.sqrt)
r_pandas = ...   # TODO: X.corr(y)

print(f"r (by hand) = {r_manual:.3f} | r (pandas) = {r_pandas:.3f}")

✅ **Self-check**

In [ ]:
assert np.isclose(r_manual, r_pandas), "the formula and .corr() must give the same result"
assert round(r_manual, 3) == 0.990, "expected r: 0.990"
print("✅ 1.2 correct")

✍️ **Question:** interpret $r$ with the course scale ($|r| < 0.30$: very weak · $0.30$ to $0.70$: moderate · $> 0.70$: strong). What do the **sign** and the **absolute value** tell you?

…

---
## Step 2 — Simple linear regression (section 2)
We model the relationship with a line: $Y = \beta_0 + \beta_1 X + \varepsilon$.

### 2.1 Ordinary least squares (OLS), by hand
$$\hat{\beta}_1 = \frac{\sum_{i=1}^{n} (x_i - \bar{x})(y_i - \bar{y})}{\sum_{i=1}^{n} (x_i - \bar{x})^2} \qquad\qquad \hat{\beta}_0 = \bar{y} - \hat{\beta}_1\,\bar{x}$$
Compute the slope `b1` and the intercept `b0` (reuse `ex` and `ey`).

In [ ]:
b1 = ...   # TODO: slope
b0 = ...   # TODO: intercept

print(f"ŷ = {b0:.3f} + {b1:.3f} X")

✅ **Self-check**

In [ ]:
assert round(b1, 3) == 0.899, "expected slope: 0.899"
assert round(b0, 3) == 4.786, "expected intercept: 4.786"
print("✅ 2.1 correct")

### 2.2 With scikit-learn
`LinearRegression()` fits the least-squares line. Fit the model, then compare with your values.

💡 scikit-learn expects a **2D array** for $X$: use double brackets `grades[["hours"]]`.

In [ ]:
model = ...   # TODO: LinearRegression()
# TODO: fit the model on grades[["hours"]] and y

a = model.intercept_   # intercept
b = model.coef_[0]     # slope
print(f"ŷ = {a:.3f} + {b:.3f} X")

✅ **Self-check**

In [ ]:
assert np.isclose(a, b0) and np.isclose(b, b1), "scikit-learn and the OLS formula must give the same line"
print("✅ 2.2 correct")

✍️ **Question:** interpret the **slope** and the **intercept** with one sentence each, in context (hours, grade).

…

### 2.3 Evaluate the model: $R^2$ (section 3)
$R^2$ = proportion of the variability of $Y$ explained by the model:
$$R^2 = 1 - \frac{\sum_{i} (y_i - \hat{y}_i)^2}{\sum_{i} (y_i - \bar{y})^2}$$
1. Compute the predictions `y_pred`, then `r2` with `model.score(...)`.
2. Recompute it **by hand** (`r2_manual`) with the formula.

In [ ]:
y_pred = ...      # TODO: model.predict(grades[["hours"]])
r2 = ...          # TODO: model.score(grades[["hours"]], y)
r2_manual = ...   # TODO: formula above

print(f"R² = {r2:.3f} | by hand = {r2_manual:.3f} | r² = {r_manual ** 2:.3f}")

✅ **Self-check**

In [ ]:
assert np.isclose(r2, r2_manual), "model.score and the formula must give the same R²"
assert np.isclose(r2, r_manual ** 2), "in simple linear regression, R² = r²"
print("✅ 2.3 correct")

✍️ **Questions**
1. Interpret the value of $R^2$ in one sentence ("… % of …").
2. What link do you observe between $R^2$ and the $r$ from step 1?

…

### 2.4 Visualise: the scatter plot and the line
*Always visualise the data and the line to validate the model.* Plot the points **and** the regression line on the same figure, with a legend.

In [ ]:
# TODO: plt.scatter(X, y, ...) for the data, plt.plot(X, y_pred, ...) for the line, then plt.legend()

### 2.5 Predict… without extrapolating
Use the model to predict the grade of students who work 5 h, 10 h, 15 h and **25 h** per week.

In [ ]:
new = pd.DataFrame({"hours": [5, 10, 15, 25]})
new["predicted_grade"] = ...   # TODO: model.predict(new[["hours"]])
new.round(2)

✅ **Self-check**

In [ ]:
assert np.allclose(new["predicted_grade"], a + b * new["hours"]), "ŷ = a + b X"
assert new["predicted_grade"].iloc[-1] > 20, "look at the prediction for 25 h..."
print("✅ 2.5 correct")

✍️ **Question:** what does the model predict for 25 h? Is it possible? Which good practice from the course did we forget?

…

### 2.6 Look at the residuals
A student's residual is the gap between the actual and the predicted grade: $e_i = y_i - \hat{y}_i$. If the linear model is suitable, the residuals are spread **at random** around 0 (linearity and homoscedasticity assumptions). Provided cell.

In [ ]:
residuals = y - y_pred

plt.figure(figsize=(7, 3))
plt.axhline(0, color="gray", linestyle="--")
plt.scatter(X, residuals, color="crimson", s=60)
plt.xlabel("Study hours per week (h)")
plt.ylabel("Residual (points)")
plt.title("Regression residuals")
plt.show()

✍️ **Question:** are the residuals spread at random around 0? What does their shape reveal, despite an $R^2$ of 0.98?

…

---
## Step 3 — The pitfalls of the coefficient $r$
*"$r$ only measures a linear relationship. […] Always analyse the context and the variables involved!"*

### 3.1 Four datasets, the same numbers: Anscombe's quartet
The four small datasets below (11 points each) are famous in statistics. For each one, compute $r$, the slope, the intercept and $R^2$.

In [ ]:
x123 = [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5]
anscombe = {
    "I":   (x123, [8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68]),
    "II":  (x123, [9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74]),
    "III": (x123, [7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73]),
    "IV":  ([8] * 7 + [19] + [8] * 3, [6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89]),
}

In [ ]:
rows = {}
for name, (xs, ys) in anscombe.items():
    xs, ys = pd.Series(xs, name="x"), pd.Series(ys, name="y")
    m = LinearRegression().fit(xs.to_frame(), ys)
    rows[name] = {
        "r": ...,           # TODO: correlation between xs and ys
        "slope": ...,       # TODO: m.coef_[0]
        "intercept": ...,   # TODO
        "R2": ...,          # TODO: m.score(...)
    }

summary = pd.DataFrame(rows).T.round(2)
summary

✅ **Self-check**

In [ ]:
assert (summary["r"] == 0.82).all(), "the four datasets have the same r"
assert (summary["slope"] == 0.5).all() and (summary["intercept"] == 3.0).all(), "... and the same line"
print("✅ 3.1 correct")

Provided cell: the four scatter plots, with the line $\hat{y} = 3 + 0.5X$.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5), sharex=True, sharey=True)
for ax, (name, (xs, ys)) in zip(axes, anscombe.items()):
    ax.scatter(xs, ys, color="royalblue")
    ax.plot([3, 20], [3 + 0.5 * 3, 3 + 0.5 * 20], color="crimson")
    ax.set_title(f"Dataset {name} — r = {summary.loc[name, 'r']}")
plt.tight_layout()
plt.show()

✍️ **Question:** for each of the four datasets, is the regression line a good model? What do you conclude?

…

### 3.2 $r \approx 0$ does not mean "no relationship"
Here $Y = X^2$: $Y$ is **entirely determined** by $X$. Compute $r$.

In [ ]:
x_u = np.linspace(-3, 3, 61)
y_u = x_u ** 2

r_u = ...   # TODO: correlation between x_u and y_u — np.corrcoef(x_u, y_u)[0, 1]
print(f"r = {r_u:.3f}")

✅ **Self-check**

In [ ]:
assert abs(r_u) < 1e-9, "r must be zero"
print("✅ 3.2 correct")

In [ ]:
plt.figure(figsize=(5, 3))
plt.scatter(x_u, y_u, s=12, color="royalblue")
plt.title(f"Y = X² — r = {abs(r_u):.2f}")
plt.show()

✍️ **Question:** how can a perfect relationship give $r = 0$?

…

---
## Step 4 — Correlation matrix on realistic data
*When several quantitative variables are studied together, the correlation matrix shows all pairwise Pearson correlations.*

### 4.0 Load the Lab 3 clients

In [ ]:
df = ...   # TODO: pd.read_csv("clients_enriched.csv", ...) — separator ";"

print(df.shape)
df.head()

✅ **Self-check**

In [ ]:
assert df.shape == (988, 11), f"Expected (988, 11), got {df.shape} -> separator?"
print("✅ 4.0 correct")

### 4.1 The matrix and its heatmap
1. Compute the correlation matrix `corr` of the five quantitative variables.
2. Display it with `sns.heatmap` (`annot=True, fmt=".2f", cmap="RdYlGn", vmin=-1, vmax=1, center=0`).

💡 `df.corr(numeric_only=True)` would also keep the `id` column, which is numeric… but it is an **identifier** (Lab 3): we choose the columns explicitly.

In [ ]:
num_cols = ["age", "income", "n_children", "tenure", "n_purchases"]

corr = ...   # TODO: correlation matrix of df[num_cols]

# TODO: heatmap

✅ **Self-check**

In [ ]:
assert corr.shape == (5, 5), "5 quantitative variables (without id)"
assert np.allclose(np.diag(corr), 1), "the diagonal equals 1"
assert np.allclose(corr, corr.T), "the matrix is symmetric"
print("✅ 4.1 correct")

### 4.2 Which relationship is the strongest?
*Focus on correlations that are high in absolute value.* Find **with code** the most correlated pair of variables (off the diagonal).

In [ ]:
off_diag = corr.where(~np.eye(len(corr), dtype=bool))   # provided: the diagonal is replaced by NaN

pair = ...    # TODO: off_diag.abs().stack().idxmax() -> (variable 1, variable 2)
r_max = ...   # TODO: value of r for this pair (corr.loc[...])
print(pair, round(r_max, 2))

✅ **Self-check**

In [ ]:
assert set(pair) == {"income", "n_purchases"}
assert round(r_max, 2) == 0.29
print("✅ 4.2 correct")

✍️ **Question:** describe this relationship with the course scale. What about the other pairs? Compare with the hours / grade example.

…

### 4.3 A single outlier is enough
*Beware of outliers, which can influence $r$.* In Lab 2, we fixed an income entered as **999,999**. Put this value back for **a single** client (out of 988) and recompute the income – number of purchases correlation.

In [ ]:
income_outlier = df["income"].copy()
income_outlier.iloc[0] = 999999

r_without = ...   # TODO: correlation between df["income"] and df["n_purchases"]
r_with = ...      # TODO: correlation between income_outlier and df["n_purchases"]
print(f"without outlier: r = {r_without:.2f} | with outlier: r = {r_with:.2f}")

✅ **Self-check**

In [ ]:
assert np.isclose(r_without, r_max)
assert r_with < r_without - 0.1, "a single point must clearly lower r"
print("✅ 4.3 correct")

✍️ **Question:** by how much does $r$ change? Which step must therefore always come before a correlation analysis?

…

### 4.4 Multicollinearity and causation
We add two variables seen in Lab 3: the **logarithm** of income and the **encoded education level** (Bachelor = 0, Master = 1, Doctorate = 2). Compute the correlation matrix of the four variables below.

In [ ]:
df["log_income"] = np.log(df["income"])
df["education_code"] = df["education"].map({"Bachelor": 0, "Master": 1, "Doctorate": 2})   # ordinal encoding (Lab 3)

corr2 = ...   # TODO: correlation matrix of ["income", "log_income", "education_code", "n_purchases"]
corr2.round(2)

✅ **Self-check**

In [ ]:
assert corr2.shape == (4, 4)
assert corr2.loc["income", "log_income"] > 0.8, "income and log_income are almost the same information"
assert 0.5 < corr2.loc["income", "education_code"] < 0.7
print("✅ 4.4 correct")

✍️ **Questions**
1. Which pair exceeds the course threshold $|r| > 0.8$? What should be done with it in a single model?
2. Education level and income are correlated ($r \approx 0.62$). Can we conclude that **getting a doctorate increases income**? Suggest at least one other explanation.

…

---
## Step 5 — A regression on realistic data
Can we predict a client's **number of purchases** from their **income**?

### 5.1 Fit and evaluate
Fit a simple linear regression `n_purchases ~ income` on the 988 clients; get the slope and $R^2$.

In [ ]:
X_c = df[["income"]]
y_c = df["n_purchases"]

model_c = ...   # TODO: LinearRegression fitted on (X_c, y_c)
slope_c = ...   # TODO: slope
r2_c = ...      # TODO: R²

print(f"purchases = {model_c.intercept_:.2f} + {slope_c:.7f} × income | R² = {r2_c:.3f}")

✅ **Self-check**

In [ ]:
assert np.isclose(r2_c, r_without ** 2), "R² = r² (step 4.3)"
assert 0 < slope_c * 10000 < 1, "how many extra purchases for 10,000 DA of extra income?"
print("✅ 5.1 correct")

Provided cell: the cloud of the 988 clients and the fitted line.

In [ ]:
plt.figure(figsize=(7, 4.5))
plt.scatter(df["income"], y_c, alpha=0.25, s=15, color="royalblue", label="Clients")
plt.plot(df["income"], model_c.predict(X_c), color="crimson", linewidth=2, label=f"Regression line (R² = {r2_c:.2f})")
plt.xlabel("Income (DA)")
plt.ylabel("Number of purchases")
plt.legend()
plt.show()

✍️ **Your analysis**
1. The slope is tiny: does that mean income has no effect? Express it for **10,000 DA** of extra income.
2. Interpret $R^2$. Does this model correctly predict the number of purchases of **a given client**? Compare with step 2.

⭐ **Bonus:** add `age`, `n_children`, `tenure` and `education_code` as explanatory variables (**multiple** regression: `df[[...]]` with several columns). Does $R^2$ increase a lot?

…

---
## Step 6 — Summary and conclusions
*Chapter goal: understand and quantify the relationship between two variables, then interpret the results obtained with Python.*

✍️ **Summary table** — fill in with your results.

| Dataset | $r$ | Equation $\hat{y} = a + bX$ | $R^2$ | Interpretation |
|---|---|---|---|---|
| Study hours → grade | … | … | … | … |
| Income → number of purchases | … | … | … | … |

✍️ **Conclusions** (5 to 10 lines): what do $r$ and $R^2$ measure? Which pitfalls did you meet? What must always be done before interpreting a coefficient?

…

---
## Summary

| Need | Tool | Remember |
|---|---|---|
| Measure a linear relationship | `s1.corr(s2)` | sign = direction, $|r|$ = strength; $-1 \le r \le 1$ |
| All pairs at once | `df[cols].corr()` + `sns.heatmap` | symmetric, diagonal = 1; $|r| > 0.8$ → multicollinearity |
| Fit a line | `LinearRegression().fit(X, y)` | `coef_[0]` = slope, `intercept_` = intercept |
| Evaluate the model | `model.score(X, y)` | $R^2$ = share of variability explained; $= r^2$ in simple regression |
| Predict | `model.predict(X_new)` | do not extrapolate outside the observed range |
| Validate | scatter plot + residuals | always visualise before interpreting |

**Golden rule:** correlation ≠ causation — and a number never replaces a plot.